# 6. Brightness and Contrast: NumPy vs. OpenCV

## Objective
Apply the linear transform output = clip(0.75 * input + 20, 0, 255) and compare the NumPy and OpenCV results.

The notebook reports the per-call median runtime, checks the maximum pixel difference, and visualizes the image, intensity histograms, and transformation curve.

## Run

Install the packages listed in the repository-level requirements.txt. Set IMAGE_PATH to a local image path to replace the sample input, then run the code cell.

In [ ]:
from pathlib import Path
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage import data

# Set this to Path("path/to/image.jpg") to use your own image.
IMAGE_PATH = None
ALPHA = 0.75
BETA = 20.0

if IMAGE_PATH is None:
    image = data.camera()
else:
    image = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

def numpy_transform(image: np.ndarray) -> np.ndarray:
    adjusted = np.rint(image.astype(np.float32) * ALPHA + BETA)
    return np.clip(adjusted, 0, 255).astype(np.uint8)

def opencv_transform(image: np.ndarray) -> np.ndarray:
    return cv2.convertScaleAbs(image, alpha=ALPHA, beta=BETA)

def median_runtime_ms(function, repeats: int = 7, iterations: int = 25) -> float:
    function()
    samples = []
    for _ in range(repeats):
        start = perf_counter()
        for _ in range(iterations):
            function()
        samples.append((perf_counter() - start) * 1000 / iterations)
    return float(np.median(samples))

numpy_result = numpy_transform(image)
opencv_result = opencv_transform(image)
difference = np.abs(numpy_result.astype(np.int16) - opencv_result.astype(np.int16))

numpy_ms = median_runtime_ms(lambda: numpy_transform(image))
opencv_ms = median_runtime_ms(lambda: opencv_transform(image))
print(f"Transform: output = clip({ALPHA} * input + {BETA}, 0, 255)")
print(f"NumPy median: {numpy_ms:.4f} ms per call")
print(f"OpenCV median: {opencv_ms:.4f} ms per call")
print(f"Maximum absolute pixel difference: {difference.max()}")
print(f"Mean intensity: {image.mean():.2f} to {numpy_result.mean():.2f}")
assert difference.max() <= 1, "NumPy and OpenCV outputs differ by more than one intensity value."

figure, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].imshow(image, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Original")
axes[0, 1].imshow(numpy_result, cmap="gray", vmin=0, vmax=255)
axes[0, 1].set_title("Transformed")
for axis in axes[0]:
    axis.axis("off")

axes[1, 0].hist(image.ravel(), bins=256, range=(0, 256), color="gray", alpha=0.7, label="Original")
axes[1, 0].hist(numpy_result.ravel(), bins=256, range=(0, 256), color="tab:blue", alpha=0.55, label="Transformed")
axes[1, 0].set_title("Intensity histograms")
axes[1, 0].set_xlabel("Intensity")
axes[1, 0].legend()

x = np.arange(256)
axes[1, 1].plot(x, x, "--", color="gray", label="Identity")
axes[1, 1].plot(x, np.clip(ALPHA * x + BETA, 0, 255), color="tab:blue", label="Transform")
axes[1, 1].set(xlim=(0, 255), ylim=(0, 255), xlabel="Input intensity", ylabel="Output intensity")
axes[1, 1].set_title("Transformation curve")
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

figure.tight_layout()
plt.show()